# Hyperparameter Tuning

Runs a Hyperopt (TPE) search over LightGBM hyperparameters for the Home Credit
default-prediction model.
Uses the same features as `02_training_LightGBM_v2.ipynb`: v1's top features by
out-of-fold SHAP (cumulative share ≤ 90%).

**Pipeline overview**
1. Load `model_dataset_final.parquet`
2. Train/validation split (fixed split reused by Notebook 3)
3. Define the search space & objective function
4. Run the Hyperopt search
5. Resolve & save the best hyperparameters
6. Save search history for diagnostics

**Output:**
- `data/Model-Training/HyperParameters/LightGBM_v2/lgbm_resolved_best.json` — best params found by the search
- `data/Model-Training/HyperParameters/LightGBM_v2/lgbm_trials_history.parquet` — full search history
- `data/Model-Training/HyperParameters/LightGBM_v2/selected_features.json` — this version's feature list, read by the training notebook

> **Note on reproducibility (Bug 17 in the original):** the original notebook ran a
> live Hyperopt search and then silently overwrote the result with a hardcoded dict
> from a previous run — discarding the live search entirely without any indication.
> This notebook keeps **both** paths but makes the active one explicit via the
> `HYPERPARAMS_SOURCE` setting in Section 1, and always prints which source is in effect.

## 1. Imports & Configuration

`HYPERPARAMS_SOURCE` picks where the hyperparameters come from:

- `'search'` (default) — run the full live Hyperopt search and save the best result.
- `'previous'` — keep the previously-saved resolved-best JSON (fast, skips the search).

Set it with `05_model_pipeline`'s `tuning` param (`HPO_HYPERPARAMS_SOURCE`
env var). The training notebook then finds the number of trees and the
classification threshold itself, by cross-validation.

In [31]:
import os
import gc; 
import json
from datetime import datetime
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from hyperopt import fmin, tpe, hp, Trials, space_eval

DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")
SRC_ROOT  = os.environ.get("HOMECREDIT_SRC_DIR", r"a:\Dominic\My Projects\Home Credit\src")
HPO_DIR  = os.path.join(DATA_ROOT, "Model-Training", "HyperParameters", "LightGBM_v2")

os.makedirs(HPO_DIR, exist_ok=True)

PATH_MODEL_DATASET = os.path.join(DATA_ROOT, "Model-Training", "ModelDataset", "model_dataset_final.parquet")
PATH_PARS_BEST = os.path.join(HPO_DIR, "lgbm_resolved_best.json")
# v1's output folder: same feature selection as 02_training_LightGBM_v2.ipynb
PATH_FEATURE_IMPORTANCE = os.path.join(DATA_ROOT, "Model-Training", "ModelOutput", "LightGBM_v1")
PATH_TRIALS_HISTORY = os.path.join(HPO_DIR, "lgbm_trials_history.csv")
PATH_FEATURE_LIST = os.path.join(HPO_DIR, "selected_features.json")  # read by the training notebook

# ── Reproducibility / split config — MUST match training exactly ────────
RANDOM_STATE = 1994
TEST_SIZE    = 0.20

# ── Where the hyperparameters come from ───────────────────────────────────
#   'search'   -> run the live Hyperopt search below and save its result (default)
#   'previous' -> keep the previously-saved resolved-best JSON (fast, skips the search)
# Set per run by 05_model_pipeline's `tuning` param (HPO_HYPERPARAMS_SOURCE env
# var) without editing the notebook; HPO_USE_PREVIOUS_BEST=true still maps to 'previous'.
# (Training with the library's default hyperparameters is a switch in the training
# notebook instead.)
HYPERPARAMS_SOURCE = os.environ.get("HPO_HYPERPARAMS_SOURCE", "").strip().lower()
if not HYPERPARAMS_SOURCE:
    HYPERPARAMS_SOURCE = ("previous" if os.environ.get("HPO_USE_PREVIOUS_BEST", "False").strip().lower()
                          in ("1", "true", "yes") else "search")
assert HYPERPARAMS_SOURCE in ("search", "previous"), f"unknown HYPERPARAMS_SOURCE {HYPERPARAMS_SOURCE!r}"
# pick a specific number of points from search space, try each one, and give me the best one you found
MAX_EVALS = 150

In [19]:
print(f"Hyperparameter source: {HYPERPARAMS_SOURCE}")

False

## 2. Load Data & Split

> **Fix (Bug 7 in the original):** the original used an 80/20 split here for HPO, but
> a *different* 85/15 split later for final training/evaluation in the training
> notebook — meaning the model selected via HPO was never actually validated on the
> exact data split used for final reporting. `RANDOM_STATE` and `TEST_SIZE` are now
> defined once above and must be reused identically in Notebook 3.

In [20]:
model_dataset = pd.read_parquet(PATH_MODEL_DATASET)
# Features: v1's most important features by out-of-fold SHAP -- every feature whose
# cumulative SHAP share (running total, most important first) is within SHAP_CUTOFF.
# Out-of-fold SHAP never touches the test set, so this choice doesn't bias v2's test metrics.
# Saved to selected_features.json (Section 6) and read by 02_training_LightGBM_v2.ipynb.
SHAP_CUTOFF = 0.90
PATH_COMBINED_IMPORTANCE = os.path.join(PATH_FEATURE_IMPORTANCE, 'feature_importance_combined.xlsx')
try:
    importance_df = pd.read_excel(PATH_COMBINED_IMPORTANCE, sheet_name='summary')
except ValueError:
    raise ValueError(f"{PATH_COMBINED_IMPORTANCE} has no 'summary' sheet -- it comes from an older "
                     "v1 run; re-run 02_training_LightGBM_v1.ipynb (v1) first.")
importance_df = importance_df.sort_values('rank')
feature_cols = importance_df.loc[importance_df['cumulative_shap_share'] <= SHAP_CUTOFF, 'feature'].tolist()
print(f"Selected {len(feature_cols)} of {len(importance_df)} v1 features "
      f"(cumulative out-of-fold SHAP share <= {SHAP_CUTOFF:.0%})")

# Re-derive cat_cols and re-apply 'category' dtype — parquet round-trips pandas
# categories correctly, but we re-assert here defensively in case the file was
# regenerated by a different process.
cat_cols = model_dataset.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    model_dataset[col] = model_dataset[col].astype('category')

X = model_dataset[['SK_ID_CURR'] + feature_cols]
y = model_dataset['TARGET']
X.set_index('SK_ID_CURR', inplace=True)

print(f"X shape: {X.shape}")
print(f"Categorical columns: {len(cat_cols)}")

FEATURE_RULE = f"v1 cumulative out-of-fold SHAP share <= {SHAP_CUTOFF:.0%}"   # recorded with the saved feature list

Selected 48 features for model training (cumulative importance <= 0.8)
X shape: (307511, 48)
Categorical columns: 10


Convert `float64` type columns to `float32`

In [21]:
# ADD THIS TO SHRINK THE DATAFRAME IN MEMORY:
float_cols = X.select_dtypes(include=['float64']).columns
X[float_cols] = X[float_cols].astype('float32')

C:\Users\domjo\AppData\Local\Temp\ipykernel_13352\3389940897.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[float_cols] = X[float_cols].astype('float32')


In [22]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"X_train: {X_train.shape}  |  X_val: {X_val.shape}")

X_train: (246008, 48)  |  X_val: (61503, 48)


In [23]:
del X, y, model_dataset
gc.collect()

1928

## 3. Define Search Space

Boosting type is fixed to `'goss'` (Gradient-based One-Side Sampling), which does
**not** use row subsampling — `subsample` therefore has zero effect under GOSS.

> **Note:** `n_estimators` is not needed as part of search space. Instead it is given a large value and controlled through `lgb.early_stopping`.

In [24]:
BOOSTING_TYPE = 'goss'  # GOSS ignores subsample — excluded from the space below

space = {
    # Core tree structure
    'num_leaves': hp.quniform('num_leaves', 20, 150, 1),
    'max_depth':  hp.quniform('max_depth', 4, 12, 1),

    # Learning rate & boosting iterations
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.2)),
  # 'n_estimators':  hp.quniform('n_estimators', 10, 1000, 10),

    # Feature sampling (GOSS handles row sampling internally — only column sampling tuned here)
    'colsample_bytree': hp.uniform('colsample_bytree', 0.1, 1.0),

    # Regularization
    'reg_alpha':  hp.loguniform('reg_alpha', np.log(1e-3), np.log(100.0)),
    'reg_lambda': hp.loguniform('reg_lambda', np.log(1e-3), np.log(100.0)),

    # Controls adding more splits/leaves to one tree. If the calculated Gain is less than your min_split_gain, LightGBM says, 
    # "This split isn't worth the effort," and refuses to split that leaf.
    'min_split_gain': hp.loguniform('min_split_gain', np.log(1e-3), np.log(1.0)),

    # Overfitting control for wide datasets. Range widened from the original
    # 20-500 -- the last live search's resolved_best pinned min_child_samples
    # at 500 (the old upper bound), a classic sign the true optimum was being
    # cut off by the search space rather than actually preferring a small leaf
    # size. Floor raised too since 20 (LightGBM's own default) is very small
    # relative to this dataset's ~250K-row training folds.
    'min_child_samples': hp.quniform('min_child_samples', 100, 10000, 20),
}

INT_PARAMS = ['num_leaves', 'max_depth', 'min_child_samples']

## 4. Objective Function

In [ ]:
def objective(params):
    """
    Hyperopt objective: 5-fold CV mean AUC for one LightGBM hyperparameter set.

    Hyperopt minimizes the return value, so we negate AUC (higher AUC = lower loss).
    Manual K-fold (rather than cross_val_score) is used so each fold can use
    LightGBM's native early stopping.
    """
    current_params = params.copy()
    for p in INT_PARAMS:
        current_params[p] = int(current_params[p])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    auc_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(
            objective='binary',
            boosting_type=BOOSTING_TYPE,
            max_bin=63,
            device='cpu',
            seed=42,
            n_jobs=14,#Leave cores available for Airflow
            n_estimators=500, #numbr of boosted trees to fit (will be limited by early stopping)
            verbose=-1,
            **current_params,
        )
        model.fit(
            X_tr, y_tr,
            eval_metric='auc',
            eval_set=[(X_va, y_va)],
            callbacks=[
                lgb.early_stopping( 
                #controls building of enough trees
                    stopping_rounds=20, #if 15 new trees are built without improvement of AUC by min_delta then stop further ensembles
                    verbose=False,
                    # first_metric_only=True, # Recommended when using min_delta
                    # min_delta=0.001 # minimum metric improvement needed to add new trees
                    )
                ],
        )
        
        preds = model.predict_proba(X_va)[:, 1]
        auc_scores.append(roc_auc_score(y_va, preds))
        # ADD THIS: Delete the fold data and force garbage collection
        del X_tr, y_tr, X_va, y_va, model
        gc.collect()


    mean_auc = np.mean(auc_scores)
    return {'loss': -mean_auc, 'status': 'ok'}

## 5. Run / Load Search

Controlled by `HYPERPARAMS_SOURCE` in Section 1. Each mode produces a single,
clearly-labelled `resolved_best` dict and prints which source produced it — there is
no silent overwrite as in the original notebook.

In [26]:
if HYPERPARAMS_SOURCE == 'previous':
    print(f"HYPERPARAMS_SOURCE='previous' → loading saved params from {PATH_PARS_BEST}")
    with open(PATH_PARS_BEST) as f:
        resolved_best = json.load(f)
    trials = None

else:
    print(f"HYPERPARAMS_SOURCE='search' → running live Hyperopt search ({MAX_EVALS} evals)")
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=MAX_EVALS,
        trials=trials,
    )
    print("\n--- Hyperopt Search Complete ---")

    resolved_best = space_eval(space, best)
    for p in INT_PARAMS:
        resolved_best[p] = int(resolved_best[p])

print(f"\nActive hyperparameter source: {HYPERPARAMS_SOURCE}")
print(json.dumps(resolved_best, indent=4) if resolved_best else "(none)")

USE_PREVIOUS_BEST=False → running live Hyperopt search (150 evals)
100%|██████████| 150/150 [2:09:04<00:00, 51.63s/trial, best loss: -0.7764386881462657]    

--- Hyperopt Search Complete ---

Active resolved_best source: live_search
{
    "colsample_bytree": 0.32233170809368833,
    "learning_rate": 0.031685497104129066,
    "max_depth": 12,
    "min_child_samples": 120,
    "min_split_gain": 0.016586271073627878,
    "num_leaves": 135,
    "reg_alpha": 8.282563247918814,
    "reg_lambda": 3.3417320214987387
}


## 6. Save Results

Writes `resolved_best` to disk only after a live search (`'previous'` leaves the
saved file untouched), along with the full trial history.

> **Fix (Bug 16 in the original):** `trials_df` / `trials_history` were computed but
> never saved — discarding useful diagnostic data about the search trajectory. Now
> saved to `lgbm_trials_history.parquet` whenever a live search runs.

In [27]:
if HYPERPARAMS_SOURCE == 'search':   # 'previous' leaves the saved file as is
    with open(PATH_PARS_BEST, 'w') as f:
        json.dump(resolved_best, f, indent=4)
        print(f"Saved resolved_best → {PATH_PARS_BEST}")

if trials is not None:
    trials_df = pd.DataFrame(trials.results)
    trials_df['auc'] = -trials_df['loss']
    trials_df['trial_number'] = range(len(trials_df))

    params_df = pd.DataFrame([t['misc']['vals'] for t in trials.trials])
    params_df = params_df.applymap(lambda x: x[0] if isinstance(x, list) and x else None)

    trials_history = pd.concat([trials_df, params_df], axis=1)
    trials_history.to_csv(PATH_TRIALS_HISTORY, index=False)
    print(f"Saved trial history ({len(trials_history)} trials) → {PATH_TRIALS_HISTORY}")

    print(f"\nBest CV AUC found: {trials_history['auc'].max():.4f}")
else:
    print("No live search ran — trial history not available this run.")

# Feature list for the training notebook, saved on every run (search or previous): training
# reads this file instead of selecting features itself, so both always use the same list
feature_list = {
    'features': list(feature_cols),
    'rule': FEATURE_RULE,
    'hyperparameters_source': HYPERPARAMS_SOURCE,
    'created_at': datetime.now().isoformat(timespec='seconds'),
}
with open(PATH_FEATURE_LIST, 'w') as f:
    json.dump(feature_list, f, indent=2)
print(f"Saved feature list ({len(feature_cols)} features; {FEATURE_RULE}) → {PATH_FEATURE_LIST}")

Saved resolved_best (live_search) → a:\Dominic\My Projects\Home Credit\data\Model-Training\HyperParameters\LightGBM_v2\lgbm_resolved_best.json
Saved trial history (150 trials) → a:\Dominic\My Projects\Home Credit\data\Model-Training\HyperParameters\LightGBM_v2\lgbm_trials_history.parquet

Best CV AUC found: 0.7764


C:\Users\domjo\AppData\Local\Temp\ipykernel_13352\1608082466.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  params_df = params_df.applymap(lambda x: x[0] if isinstance(x, list) and x else None)


## 7. Cleanup

In [28]:
del X_train, X_val, y_train, y_val
gc.collect()
print("All DataFrames released.")

All DataFrames released.
